# Events and Sigma-Algebras

Concept 18 of the math-foundations learning map. Companion to `README.md` and `lesson.tex`.

Goal of this notebook: build small, finite witnesses of the sigma-algebra axioms so the abstract definitions feel concrete.

## 1. Setup

We work with $\Omega = \{1,2,3,4\}$. On a finite carrier, countable union collapses to finite union, so we can verify the sigma-algebra axioms by exhaustive enumeration.

In [ ]:
from itertools import combinations

def powerset(s):
    items = list(s)
    return frozenset(
        frozenset(c)
        for r in range(len(items) + 1)
        for c in combinations(items, r)
    )

omega = {1, 2, 3, 4}
P = powerset(omega)
len(P)

## 2. Verifying the three axioms

Recall: $\mathcal{F}$ is a sigma-algebra on $\Omega$ iff

1. $\Omega \in \mathcal{F}$,
2. $A \in \mathcal{F} \Rightarrow A^c \in \mathcal{F}$,
3. $A_1, A_2, \ldots \in \mathcal{F} \Rightarrow \bigcup A_n \in \mathcal{F}$.

In [ ]:
def is_sigma_algebra(F, omega):
    F = set(F)
    omega = frozenset(omega)
    if omega not in F:
        return False, 'Omega missing'
    for A in F:
        if (omega - A) not in F:
            return False, f'complement of {set(A)} missing'
    for A in F:
        for B in F:
            if (A | B) not in F:
                return False, f'union {set(A)} U {set(B)} missing'
    return True, 'OK'

is_sigma_algebra(P, omega)

## 3. The trivial sigma-algebra

$\mathcal{F}_0 = \{\emptyset, \Omega\}$ is the smallest sigma-algebra on any $\Omega$.

In [ ]:
F0 = {frozenset(), frozenset(omega)}
is_sigma_algebra(F0, omega)

## 4. Sigma-algebra generated by an event

Closing $\{A\}$ under complement and finite union should give exactly $\{\emptyset, A, A^c, \Omega\}$ when $A \ne \emptyset, \Omega$.

In [ ]:
def generated(generators, omega):
    omega = frozenset(omega)
    F = {frozenset(), omega} | {frozenset(g) for g in generators}
    changed = True
    while changed:
        changed = False
        new = set(F)
        for A in F:
            new.add(omega - A)
        for A in F:
            for B in F:
                new.add(A | B)
        if new != F:
            F = new
            changed = True
    return F

G = generated([{1, 2}], omega)
sorted([sorted(s) for s in G], key=lambda x: (len(x), x))

## 5. Two generators: $\sigma(\{A, B\})$

With $A = \{1,2\}$ and $B = \{2,3\}$, the generated sigma-algebra partitions $\Omega$ into the atoms $\{1\}, \{2\}, \{3\}, \{4\}$ (singletons), so $\sigma(\{A,B\}) = 2^{\Omega}$ and has 16 elements.

In [ ]:
G2 = generated([{1, 2}, {2, 3}], omega)
print('size:', len(G2))
print('equals full power set?', G2 == P)
print('is sigma-algebra?', is_sigma_algebra(G2, omega))

## 6. Intersection of sigma-algebras is a sigma-algebra

The key structural theorem (proved in `lesson.tex`). We test it on two non-trivial sigma-algebras.

In [ ]:
F_A = generated([{1, 2}], omega)        # {empty, {1,2}, {3,4}, Omega}
F_B = generated([{1, 3}], omega)        # {empty, {1,3}, {2,4}, Omega}
inter = F_A & F_B
print('intersection:', sorted([sorted(s) for s in inter], key=lambda x: (len(x), x)))
print('is sigma-algebra?', is_sigma_algebra(inter, omega))

The intersection collapses to the trivial sigma-algebra $\{\emptyset, \Omega\}$ -- the only events that *both* refine.

## 7. Takeaways

- Sigma-algebras encode "which questions probability can answer."
- On finite carriers, every sigma-algebra is determined by a partition into atoms.
- Generation $\sigma(\mathcal{C})$ is well-defined precisely because arbitrary intersections of sigma-algebras stay sigma-algebras.
- Conditional expectation, filtrations, and martingales (concepts 27+) all build on these primitives.